In [1]:
import torch
from sorl.gat_sim import GAT, GATConfig, BOS_TOKEN_ID
torch.set_float32_matmul_precision('high')  # Enable TF32 for ~2x speedup
# For fast training, set 'BOS_TOKEN_ID' to 15 in 'sorl/gat_sim.py' 

gat_config = GATConfig(
    vocab_sizes=[BOS_TOKEN_ID+1, 6],  # 6 abstract tokens
    n_layer=4,
    n_head=4,
    n_embd=128,
    device="cuda" if torch.cuda.is_available() else "cpu",
    flex_kernel_options={
            "BLOCK_M": 32, "BLOCK_N": 32,
            "BLOCK_M1": 32, "BLOCK_N1": 64, "BLOCK_M2": 64, "BLOCK_N2": 32
        }
)
    
model = GAT(gat_config)
# model = model.to("cuda")
# model = torch.compile(model)

In [2]:
# ---- Copy & Paste Data Loader ----
from data.copy_paste import CopyPasteDataLoader

seq_len = 1
loader = CopyPasteDataLoader(vocab_size=16, max_token=10, seq_len=seq_len, device='cpu')
tokens, loss_mask = loader.get_batch(2)

tokens, loss_mask

(tensor([[15,  1, 16,  1, 15,  2, 16,  2]]),
 tensor([[0., 1., 1., 0., 0., 1., 1., 0.]]))

In [10]:
from sorl.neo_utils import sorl_search, compute_loss, sorl_evaluate

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
batch_size = 16
memory_span = 1
attn_blocksize = 1792
K = 4
max_iterations = 0
temperature = 2.0
loss_mask = None

for step in range(100): 
    optimizer.zero_grad()

    tokens, _ = loader.get_batch(batch_size)
    # tokens = next(train_loader)

    # --- mixture of SoRL selection & deep supervision (avg. loss per iteration) ---
    with torch.no_grad(): 
        search_tokens, search_ppt, search_adv = sorl_search(tokens, model, n=2, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperature, loss_mask=loss_mask)
    
    # --- compute loss ---
    traj_loss, abs_loss = compute_loss(search_tokens, model, memory_span=memory_span, attn_blocksize=attn_blocksize, loss_mask=loss_mask)
    # loss = traj_loss + abs_loss
    loss = traj_loss

    # GAPT
    
    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    if step % 2 == 0: 
        with torch.no_grad(): 
            val_tokens, val_adv, traj_loss, abs_loss = sorl_evaluate(tokens, model, n=4, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=10.0, loss_mask=loss_mask)
        print(f"validation step {step} | traj_loss: {traj_loss.item():.2f} | abs_loss: {abs_loss.item():.2f} | search adv: {val_adv.item() * 100:.2f}%")

validation step 0 | traj_loss: 1.01 | abs_loss: 3.09 | search adv: 0.00%
validation step 2 | traj_loss: 0.94 | abs_loss: 3.13 | search adv: 0.00%
validation step 4 | traj_loss: 0.90 | abs_loss: 3.23 | search adv: 0.00%
validation step 6 | traj_loss: 0.90 | abs_loss: 3.33 | search adv: 0.00%
validation step 8 | traj_loss: 0.86 | abs_loss: 3.44 | search adv: 0.00%
validation step 10 | traj_loss: 0.85 | abs_loss: 3.54 | search adv: 0.00%
validation step 12 | traj_loss: 0.83 | abs_loss: 3.63 | search adv: 0.00%
validation step 14 | traj_loss: 0.80 | abs_loss: 3.76 | search adv: 0.00%
validation step 16 | traj_loss: 0.84 | abs_loss: 3.89 | search adv: 0.00%
validation step 18 | traj_loss: 0.80 | abs_loss: 3.96 | search adv: -0.00%
validation step 20 | traj_loss: 0.83 | abs_loss: 4.11 | search adv: -0.00%
validation step 22 | traj_loss: 0.82 | abs_loss: 4.09 | search adv: 0.00%
validation step 24 | traj_loss: 0.77 | abs_loss: 3.78 | search adv: -0.00%
validation step 26 | traj_loss: 0.82 | a

In [20]:
# generate function implementation 
# ----------------------------------
from sorl.gat_sim import generate

tokens, loss_mask = loader.get_batch(1)
idx = tokens[:, :1 + seq_len].clone()
print(f"init   | idx: {idx[0].tolist()}")
for i in range(seq_len+1): 
    idx = generate(model, idx, K=K, max_iterations=0, memory_span=memory_span, attn_blocksize=1792, temperature=0.0)
    print(f"step {i+1} idx: {idx[0].tolist()}")

init   | idx: [15, 2]
step 1 idx: [15, 2, 16]
step 2 idx: [15, 2, 16, 2]


In [ ]:
# heuristic rollout
import os, glob, itertools
from pathlib import Path

# MPS specific data loader functional (single device ver.)
# --------------------------------------------
def _load_data_shard(file: Path):
    header = torch.from_file(str(file), False, 256, dtype=torch.int32) # header is 256 int32
    assert header[0] == 20240520, "magic number mismatch in the data .bin file"
    assert header[1] == 1, "unsupported version"
    num_tokens = int(header[2]) # number of tokens (claimed)
    with file.open("rb", buffering=0) as f:
        tokens = torch.empty(num_tokens, dtype=torch.uint16, pin_memory=False) # MPS requires pin_memory=False
        f.seek(256 * 4)
        nbytes = f.readinto(tokens.numpy()) # avoid bytes->array copy by @YouJiacheng
        assert nbytes == 2 * num_tokens, "number of tokens read does not match header"
    return tokens

def data_generator(filename_pattern: str, sequence_length: int, device: str): 

    filename_pattern = "data/fineweb10B/fineweb_train_*.bin"
    files = [Path(file) for file in sorted(glob.glob(filename_pattern))]
    file_iter = itertools.cycle(files)
    tokens, pos = _load_data_shard(next(file_iter)), 0
    while True: 
        # Concern 1. Doesn't this means end-of-file is never reached?
        if pos + sequence_length + 1 >= len(tokens): # not enough data left -> load a new file
            tokens, pos = _load_data_shard(next(file_iter)), 0

        idx = tokens[pos : pos + sequence_length + 1].unsqueeze(0).to(device=device, dtype=torch.int32, non_blocking=True)
        pos += sequence_length
        yield idx

# ------------------------------------------------

# Question 1. Should we separate inputs / targets? 
#             That's really asking whether we want to 'reflect' on inputs, or inputs + next token
#             from the generation perspective, we ought to reflect on inputs and predict next-tok

# Reflection 1. 
# - based on above thought, we ought to modify the 'forward' method to take 'inputs' & 'targets' separately
#   the ._forward_pass and recursion should be done only on 'inputs'


data = _load_data_shard(Path("data/fineweb10B/fineweb_train_000002.bin"))

train_loader = data_generator(filename_pattern="data/fineweb10B/fineweb_train_*.bin", sequence_length=256, device="cpu")
val_loader = data_generator(filename_pattern="data/fineweb10B/fineweb_val_000000.bin", sequence_length=256, device="cpu")

In [ ]:
# --- Benchmark Speed & Memory Cost --- 
from sorl.benchmark import run_benchmark_suite
import torch 


# Prepare data - TEST WITH SMALLER SEQUENCE FIRST
tokens = next(train_loader)

# Run benchmark
results = run_benchmark_suite(
    model, 
    tokens, 
    memory_span=1024, 
    num_runs=10
)